## Context Encoders: Inpainting in FlickrFacesHQ

This notebook contains an implementation of Inpainting using [Context Encoders](https://arxiv.org/pdf/1604.07379.pdf). 

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import os
import subprocess
import zipfile
import glob
from PIL import Image

In [ ]:
## PyTorch
import torch
import torch.nn as nn
import torch.utils.data as data
from torch.optim import Adam
import torch.autograd as autograd
from torch.autograd import Variable
from torch.nn.utils.parametrizations import spectral_norm
from torch.utils.data import Dataset
# Torchvision
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid, save_image
import torchvision.transforms.functional as F
# tqdm
from tqdm.notebook import tqdm, trange
from torchsummary import summary

In [ ]:
# Configure environment
torch.backends.cudnn.benchmark = True

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

### Load and Prepare CelebA Dataset

In [ ]:
train_dir = 'data/flickrfacesHQ/train/'
test_dir = 'data/flickrfacesHQ/test/'

In [ ]:
import os
os.makedirs("ConEncFlickrFacesHQ", exist_ok=True)

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, root, transforms_=None):
        self.transform = transforms_
        self.files = sorted(glob.glob(f"{root}/*.png"))

    def __getitem__(self, index):
        img = Image.open(self.files[index % len(self.files)])
        img = self.transform(img)
        return img

    def __len__(self):
        return len(self.files)

In [ ]:
transform = transforms.Compose([transforms.Resize((128, 128), 
                                transforms.InterpolationMode.BICUBIC),
                                transforms.ToTensor()])

In [ ]:
train_batch_size = 256
train_loader = data.DataLoader(CustomImageDataset(train_dir, transforms_=transform), 
                               batch_size=train_batch_size, 
                               shuffle=True, drop_last=True, pin_memory=True,
                               num_workers=4)

In [ ]:
test_batch_size = 16
test_loader = data.DataLoader(CustomImageDataset(test_dir, transforms_=transform),
                               batch_size=test_batch_size, 
                               shuffle=False,
                               num_workers=1)

In [ ]:
def apply_random_mask(img, img_size, mask_size):
    y1, x1 = np.random.randint(0, img_size - mask_size, 2)
    y2, x2 = y1 + mask_size, x1 + mask_size
    masked_part = img[:, :, y1:y2, x1:x2]
    masked_img = img.clone()
    masked_img[:, :, y1:y2, x1:x2] = 1
    return masked_img, masked_part, (x1, y1, x2, y2)

In [ ]:
def reconstruct_imgs(masked_imgs, gen_parts, coords):
    x1, y1, x2, y2 = coords
    reconst_img = masked_imgs.clone()
    reconst_img[:, :, y1:y2, x1:x2] = gen_parts
    return reconst_img

In [ ]:
def show_tensor_images(image_tensor, 
                       num_images=16, 
                       nrow=4,
                       filename=""):
    image_grid = make_grid(image_tensor[:num_images].detach().cpu(), nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### Build the Generator class
The generator of the Context Encoder model has an encoder-decoder structure with no random latent space.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.input = nn.Sequential(nn.Conv2d(3, 64, 4, 2, 1), 
                                   nn.LeakyReLU(negative_slope=0.2))
        self.down1 = self.downsample_block(64, 64)
        self.down2 = self.downsample_block(64, 128)
        self.down3 = self.downsample_block(128, 256)
        self.down4 = self.downsample_block(256, 512)
        self.centre = nn.Conv2d(512, 4000, 1)
        self.up4 = self.upsample_block(4000, 512)
        self.up3 = self.upsample_block(512, 256)
        self.up2 = self.upsample_block(256, 128)
        self.up1 = self.upsample_block(128, 64)
        self.output = nn.Sequential(nn.Conv2d(64, 3, 1, 1), 
                                    nn.Sigmoid())
        
    def downsample_block(self, in_channels, out_channels):
        block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 4, 2, 1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(negative_slope=0.2)
        )
        return block
    
    def upsample_block(self, in_channels, out_channels):
        block = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(negative_slope=0.2)
        )
        return block
    
    def forward(self, image):
        out = self.input(image)
        out = self.down1(out)
        out = self.down2(out)
        out = self.down3(out)
        out = self.down4(out)
        out = self.centre(out)
        out = self.up4(out)
        out = self.up3(out)
        out = self.up2(out)
        out = self.up1(out)
        return self.output(out)

### Build the Discriminator class

In [ ]:
class Discriminator(nn.Module):    
    def __init__(self):
        super(Discriminator, self).__init__()
        self.input = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1),
            nn.LeakyReLU(negative_slope=0.2)
        )
        self.down1 = self.downsample_block(64, 128)
        self.down2 = self.downsample_block(128, 256)
        self.down3 = self.downsample_block(256, 512)
        self.output = nn.Conv2d(512, 1, 4, 1, 0)
        
    def downsample_block(self, in_channels, out_channels, stride = 2):
        block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1),
            nn.InstanceNorm2d(out_channels),
            nn.LeakyReLU(negative_slope=0.2)
        )
        return block
        
    def forward(self, image):
        out = self.input(image)
        out = self.down1(out)
        out = self.down2(out)
        out = self.down3(out)
        return self.output(out)

### Train the Model

In [ ]:
adv_loss = torch.nn.MSELoss()
pix_loss = torch.nn.L1Loss()

In [ ]:
n_epochs = 200
lr = 0.0002
beta_1 = 0.5
beta_2 = 0.999
epoch_show = 5
show_batch = 16
mask_size = 64

In [ ]:
gen = Generator().to(device)
gen_opt = Adam(gen.parameters(), lr=lr, betas=(beta_1, beta_2))
disc = Discriminator().to(device) 
disc_opt = Adam(disc.parameters(), lr=lr, betas=(beta_1, beta_2))

In [ ]:
summary(gen, input_size = (3, 128, 128), batch_size=-1)

In [ ]:
summary(disc, input_size = (3, 64, 64), batch_size=-1)

In [ ]:
adv_loss_lst = list()
disc_loss_lst = list()
pixel_loss_lst = list()
tqdm_epoch = trange(n_epochs)
for epoch in tqdm_epoch:
    avg_adv_loss = 0.0
    avg_disc_loss = 0.0
    avg_pixel_loss = 0.0
    num_items = 0
    for real_imgs in tqdm(train_loader):
        cur_batch_size = real_imgs.shape[0]
        real_imgs = real_imgs.to(device)
        masked_imgs, masked_pts, coords = apply_random_mask(real_imgs, real_imgs.shape[-1], mask_size)

        valid = torch.ones(cur_batch_size, 1, 1, 1, device=device, requires_grad=False)
        fake = torch.zeros(cur_batch_size, 1, 1, 1, device=device, requires_grad=False)

        #  Train Generator
        gen_opt.zero_grad()
        genpart_imgs = gen(masked_imgs)
        validity = disc(genpart_imgs)
        galoss = adv_loss(validity, valid)
        gploss = pix_loss(genpart_imgs, masked_pts)
        gloss = 0.001 * galoss + 0.999 * gploss
        
        gloss.backward()
        gen_opt.step()
        
        # Train Discriminator
        disc_opt.zero_grad()
        real_pred = disc(masked_pts)
        fake_pred = disc(genpart_imgs.detach())
        real_loss = adv_loss(real_pred, valid)
        fake_loss = adv_loss(fake_pred, fake)
        dloss = 0.5 * (real_loss + fake_loss)
        dloss.backward()
        disc_opt.step()
        
        avg_adv_loss += galoss.item() * cur_batch_size
        avg_pixel_loss += gploss.item() * cur_batch_size
        avg_disc_loss += dloss.item() * cur_batch_size
        num_items += cur_batch_size
        
        # reconstruct images
        reconst_imgs = reconstruct_imgs(masked_imgs, genpart_imgs, coords)
        
    if (epoch + 1) % epoch_show == 0:
        
        show_tensor_images(reconst_imgs, filename='ConEncFlickrFacesHQ/ConEncFlickrFacesHQGrid_' + str(epoch) + '.png')
        show_tensor_images(real_imgs)
        
    ave_adv_loss = avg_adv_loss / num_items
    ave_pixel_loss = avg_pixel_loss / num_items
    ave_disc_loss = avg_disc_loss / num_items
    adv_loss_lst.append(ave_adv_loss)
    disc_loss_lst.append(ave_disc_loss)
    pixel_loss_lst.append(ave_pixel_loss)
    tqdm_epoch.set_description('Ave Adv Loss: {:5f}, \
                               Ave Disc Loss: {:5f}, \
                               Ave Pixel Loss: {:5f}'.format(ave_adv_loss, ave_disc_loss, ave_pixel_loss))
    torch.save(gen.state_dict(), 'conencffaces_genckpt.pth')
    torch.save(disc.state_dict(), 'conencffaces_discckpt.pth')

### Plot Losses

In [ ]:
epochs = range(1, n_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
ax.plot(epochs, adv_loss_lst, label='Adversarial Loss')
ax.plot(epochs, disc_loss_lst, label='Discriminator Loss')
ax.plot(epochs, pixel_loss_lst, label='Reconstruction Loss')
    
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.set_yscale('log')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('ConEncffacesLosses.png', dpi=300, bbox_inches='tight')

### Plot Images

In [ ]:
it = iter(test_loader)


In [ ]:
sample_img = next(it)

In [ ]:
masked_imgs, masked_pts, coords = apply_random_mask(sample_img, sample_img.shape[-1], mask_size)

In [ ]:
recon_imgs = gen(masked_imgs.to(device))

In [ ]:
plt.figure(figsize=(12,12))
show_tensor_images(sample_img, num_images=8, 
                       nrow=4, filename='ConEncffacesOrigGrid.png')

In [ ]:
plt.figure(figsize=(12,12))
show_tensor_images(masked_imgs, num_images=8, 
                       nrow=4, filename='ConEncffacesMaskGrid.png')

In [ ]:
plt.figure(figsize=(12,12))
show_tensor_images(reconstruct_imgs(masked_imgs, recon_imgs, coords), num_images=8, 
                       nrow=4, filename='ConEncffacesReconGrid.png')